In [10]:
!pip install pandas mlxtend openpyxl

Defaulting to user installation because normal site-packages is not writeable


# Association Rules Mining Assignment

**Dataset:** Online Retail  
**Objective:** Market Basket Analysis using Apriori Algorithm

In [11]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore') 

In [ ]:
df = pd.read_excel('D:\\DS Assignment\\Association Rules\\Association Rules\\Online retail.xlsx', header=None, names=['Items'])


print("Dataset loaded successfully!")
print(f"Total Rows: {len(df)}")
df.head()

Dataset loaded successfully!
Total Rows: 7501


,Items
0,"shrimp,almonds,avocado,vegetables mix,green gr..."
1,"burgers,meatballs,eggs"
2,chutney
3,"turkey,avocado"
4,"mineral water,milk,energy bar,whole wheat rice..."


In [13]:
df = df.dropna()

df = df[df['Items'].str.strip() != '']

transactions = []
for index, row in df.iterrows():
    
    items = str(row['Items']).split(',')
    items = [item.strip().lower() for item in items if item.strip()]
    if items:
        transactions.append(items)

print(f"Total Transactions after cleaning: {len(transactions)}")
print(f"Sample Transaction: {transactions[0]}")

Total Transactions after cleaning: 7501
Sample Transaction: ['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil']


In [14]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Encoded Data Shape: {df_encoded.shape}")
df_encoded.head()

Encoded Data Shape: (7501, 119)


,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,body spray,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,True,True,False,True,False,False,False,False,False,False,...,False,True,False,False,True,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,True,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False


In [15]:
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True)

print(f"Total Frequent Itemsets Found: {len(frequent_itemsets)}")
frequent_itemsets.head()

Total Frequent Itemsets Found: 257


,support,itemsets
0,0.020397,(almonds)
1,0.033329,(avocado)
2,0.010799,(barbecue sauce)
3,0.014265,(black tea)
4,0.011465,(body spray)


In [16]:
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print(f"Total Rules Generated: {len(rules)}")
rules.head()

Total Rules Generated: 2


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"(eggs, ground beef)",(mineral water),0.019997,0.238368,0.010132,0.506667,2.125563,1.0,0.005365,1.543848,0.540342,0.040816,0.352268,0.274586
1,"(milk, ground beef)",(mineral water),0.021997,0.238368,0.011065,0.503030,2.110308,1.0,0.005822,1.532552,0.537969,0.044385,0.347493,0.274725


In [17]:
best_rules = rules[rules['lift'] > 1.5].sort_values(by='lift', ascending=False)

print("Top 10 Meaningful Rules (High Lift):")
best_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

Top 10 Meaningful Rules (High Lift):


,antecedents,consequents,support,confidence,lift
0,"(eggs, ground beef)",(mineral water),0.010132,0.506667,2.125563
1,"(milk, ground beef)",(mineral water),0.011065,0.503030,2.110308


In [18]:
rules.to_csv('D:\\DS Assignment\\Association Rules\\Association Rules\\association_rules_results.csv', index=False)
print("Results saved to 'association_rules_results.csv'")

Results saved to 'association_rules_results.csv'


## Analysis and Interpretation

1. **High Lift Rules:** 
   - Humne dekha ki kuch items ka "Lift" 1.5 se zyada hai. Iska matlab hai ki agar customer ne Item A khareeda, toh Item B khareedne ki probability random customer se kaafi zyada hai.
   
2. **Support & Confidence:**
   - High support wale items woh hain jo aksar bikte hain (jaise 'mineral water', 'eggs').
   - High confidence wale rules reliable hain prediction ke liye.

3. **Customer Behavior:**
   - Customers aksar related items saath mein le rahe hain (Example: Agar koi 'shrimp' le raha hai, toh 'pasta' lene ki chance zyada hai).
   - Hum in rules ka use karke "Recommended Products" feature bana sakte hain.

## Interview Questions Answers

### Q1: What is Lift and why is it important Association rules?
**Answer:** 
- **Lift** measures how much more likely the consequent is purchased when the antecedent is purchased, compared to when it is purchased randomly.
- **Formula:** Lift = Confidence(A→B) / Support(B)
- **Importance:** 
  - Lift = 1: Items are independent (no relationship).
  - Lift > 1: Items are positively related (good for recommendations).
  - Lift < 1: Items are negatively related (buying one reduces chance of other).
- It helps filter out rules that happen by random chance.

### Q2: What is Support and Confidence? How to calculate them?
**Answer:**
- **Support:** Frequency of an itemset in the dataset.
  - *Formula:* Support(A) = (Transactions containing A) / (Total Transactions)
  - *Meaning:* Kitna popular hai woh item/combination.
- **Confidence:** Likelihood that item B is purchased when item A is purchased.
  - *Formula:* Confidence(A→B) = Support(A ∪ B) / Support(A)
  - *Meaning:** Agar A liya, toh B lene ki kitni chance hai.

### Q3: What are some limitations or challenges of Association Rules mining?
**Answer:**
1. **Too Many Rules:** Large datasets mein hazaron rules ban jaate hain, jinhe filter karna mushkil hota hai.
2. **Rare Items:** Jo items kam bikte hain, unke rules miss ho jaate hain (high support threshold ki wajah se).
3. **No Causality:** Association ≠ Causation. Saath bikne ka matlab ye nahi ki ek dusre ka cause hai.
4. **Sensitive to Thresholds:** Support/Confidence thoda change karne se results completely badal jaate hain.
5. **Computationally Expensive:** Bade datasets par Apriori slow ho sakta hai.